# Chapter 6.1: VAE for Collaborative Filtering

## Learning Objectives

By the end of this notebook, you will be able to:

1. **Understand** the Variational Autoencoder (VAE) framework and its application to collaborative filtering
2. **Derive** the Evidence Lower Bound (ELBO) for implicit feedback recommendation
3. **Implement** Mult-VAE from scratch with multinomial likelihood
4. **Explain** the RecVAE approach with composite prior and alternating training
5. **Apply** MacridVAE for disentangling macro and micro user intents
6. **Control** disentanglement in recommendations using Beta-VAE
7. **Tune** the beta hyperparameter and analyze its impact on recommendation quality

## Prerequisites

- Familiarity with PyTorch and basic neural network training
- Understanding of autoencoders and latent variable models
- Knowledge of collaborative filtering basics (Part 1)
- Basic probability theory (KL divergence, variational inference)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/USERNAME/rec_system/blob/main/notebooks/part6/chapter_6.1_vae_cf.ipynb)
[![Download](https://img.shields.io/badge/Download-Notebook-blue)](https://raw.githubusercontent.com/USERNAME/rec_system/main/notebooks/part6/chapter_6.1_vae_cf.ipynb)

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
from collections import defaultdict

# Reproducibility
np.random.seed(42)
torch.manual_seed(42)

# Device
device = torch.device('cpu')
print(f'Using device: {device}')

## 1. Background: VAE for Recommendation

Variational Autoencoders (VAEs) provide a principled probabilistic framework for collaborative filtering. Unlike traditional matrix factorization that learns point estimates, VAEs learn a **distribution** over user representations in a latent space.

### The Generative Story

For each user $u$:
1. Sample a latent representation: $\mathbf{z}_u \sim \mathcal{N}(0, \mathbf{I}_K)$
2. Generate interaction probability: $\pi(\mathbf{z}_u) = \text{softmax}(f_\theta(\mathbf{z}_u))$
3. Sample interactions: $\mathbf{x}_u \sim \text{Mult}(N_u, \pi(\mathbf{z}_u))$

The key insight of **Mult-VAE** (Liang et al., 2018, Netflix/Google) is using a **multinomial likelihood** instead of Gaussian or Bernoulli, which better models implicit feedback where we observe click counts.

### ELBO Derivation for Implicit Feedback

Since the true posterior $p(\mathbf{z}_u | \mathbf{x}_u)$ is intractable, we approximate it with $q_\phi(\mathbf{z}_u | \mathbf{x}_u)$. The Evidence Lower Bound (ELBO) is:

$$\mathcal{L}(\theta, \phi; \mathbf{x}_u) = \mathbb{E}_{q_\phi(\mathbf{z}_u|\mathbf{x}_u)}[\log p_\theta(\mathbf{x}_u|\mathbf{z}_u)] - \text{KL}(q_\phi(\mathbf{z}_u|\mathbf{x}_u) \| p(\mathbf{z}_u))$$

Where:
- **Reconstruction term**: $\mathbb{E}_{q_\phi}[\log p_\theta(\mathbf{x}_u|\mathbf{z}_u)]$ encourages accurate reconstruction
- **KL term**: $\text{KL}(q_\phi \| p)$ regularizes the latent space toward the prior

For multinomial likelihood:
$$\log p_\theta(\mathbf{x}_u | \mathbf{z}_u) = \sum_{i=1}^{I} x_{ui} \log \pi_i(\mathbf{z}_u)$$

> **💡 Concept:** The multinomial likelihood naturally handles the "bag-of-items" view of user interactions. Each user's history is treated as a multinomial draw from a user-specific distribution over items.

## 2. Synthetic Data Generation

Let's generate synthetic implicit feedback data for our experiments.

In [ ]:
def generate_synthetic_data(n_users=2000, n_items=500, n_factors=10, sparsity=0.05, seed=42):
    """Generate synthetic implicit feedback data from a latent factor model."""
    rng = np.random.RandomState(seed)
    
    # Latent factors
    user_factors = rng.randn(n_users, n_factors) * 0.5
    item_factors = rng.randn(n_items, n_factors) * 0.5
    
    # Item popularity bias
    item_bias = rng.randn(n_items) * 0.3
    
    # Interaction probabilities
    logits = user_factors @ item_factors.T + item_bias[None, :]
    probs = 1.0 / (1.0 + np.exp(-logits))
    
    # Scale to get desired sparsity
    threshold = np.quantile(probs, 1 - sparsity)
    interactions = (probs > threshold).astype(np.float32)
    
    # Add some noise
    noise_mask = rng.random(interactions.shape) < 0.01
    interactions = np.where(noise_mask, 1 - interactions, interactions)
    
    # Train/test split per user
    train_data = interactions.copy()
    test_data = np.zeros_like(interactions)
    
    for u in range(n_users):
        items = np.where(interactions[u] > 0)[0]
        if len(items) > 1:
            n_test = max(1, int(len(items) * 0.2))
            test_items = rng.choice(items, n_test, replace=False)
            train_data[u, test_items] = 0
            test_data[u, test_items] = 1
    
    return train_data, test_data

train_data, test_data = generate_synthetic_data()
print(f'Train data shape: {train_data.shape}')
print(f'Train sparsity: {train_data.mean():.4f}')
print(f'Test sparsity: {test_data.mean():.4f}')
print(f'Avg items per user (train): {train_data.sum(axis=1).mean():.1f}')
print(f'Avg items per user (test): {test_data.sum(axis=1).mean():.1f}')

## 3. Mult-VAE Implementation

The Mult-VAE architecture consists of:
- **Encoder**: Maps user interaction vector to parameters of the approximate posterior $q_\phi(\mathbf{z}|\mathbf{x})$
- **Decoder**: Maps latent vector to a distribution over items via softmax (multinomial logits)

> **🔑 Pro Tip:** Liang et al. (2018) found that a simple architecture with a single hidden layer in the encoder works surprisingly well. Deeper networks didn't help much for CF.

In [ ]:
class MultVAE(nn.Module):
    """Mult-VAE: Variational Autoencoder with Multinomial Likelihood for CF.
    
    Reference: Liang et al., "Variational Autoencoders for Collaborative Filtering", WWW 2018.
    """
    
    def __init__(self, n_items, hidden_dim=256, latent_dim=64, dropout=0.5):
        super().__init__()
        self.n_items = n_items
        self.latent_dim = latent_dim
        self.dropout = nn.Dropout(dropout)
        
        # Encoder: x -> h -> (mu, logvar)
        self.encoder = nn.Sequential(
            nn.Linear(n_items, hidden_dim),
            nn.Tanh()
        )
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)
        
        # Decoder: z -> logits over items
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, n_items)
        )
        
        self._init_weights()
    
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                nn.init.zeros_(m.bias)
    
    def encode(self, x):
        """Encode input to latent distribution parameters."""
        # Normalize input (important for multinomial interpretation)
        x_norm = F.normalize(x, p=2, dim=1)
        x_drop = self.dropout(x_norm)
        
        h = self.encoder(x_drop)
        mu = self.fc_mu(h)
        logvar = self.fc_logvar(h)
        return mu, logvar
    
    def reparameterize(self, mu, logvar):
        """Reparameterization trick: z = mu + std * epsilon."""
        if self.training:
            std = torch.exp(0.5 * logvar)
            eps = torch.randn_like(std)
            return mu + std * eps
        return mu  # Use mean at test time
    
    def decode(self, z):
        """Decode latent vector to item logits."""
        return self.decoder(z)
    
    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        logits = self.decode(z)
        return logits, mu, logvar


def mult_vae_loss(logits, x, mu, logvar, beta=1.0):
    """Compute the Mult-VAE ELBO loss.
    
    Args:
        logits: Decoder output (unnormalized log-probs)
        x: Input interaction vector
        mu: Mean of approximate posterior
        logvar: Log-variance of approximate posterior
        beta: Weight on KL divergence term
    
    Returns:
        Negative ELBO loss
    """
    # Multinomial log-likelihood (cross-entropy with softmax)
    log_softmax = F.log_softmax(logits, dim=1)
    recon_loss = -torch.sum(x * log_softmax, dim=1).mean()
    
    # KL divergence: KL(q(z|x) || p(z)) where p(z) = N(0,I)
    kl_loss = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1).mean()
    
    return recon_loss + beta * kl_loss, recon_loss, kl_loss


# Instantiate model
n_items = train_data.shape[1]
model = MultVAE(n_items=n_items, hidden_dim=256, latent_dim=64, dropout=0.5).to(device)
print(f'Model parameters: {sum(p.numel() for p in model.parameters()):,}')
print(model)

## 4. Training with KL Annealing

A key training trick for Mult-VAE is **KL annealing**: gradually increasing $\beta$ from 0 to 1 during training. This prevents the model from collapsing to the prior early in training (the "posterior collapse" problem).

> **⚠️ Common Pitfall:** Without KL annealing, the model may ignore the latent code and produce the same recommendations for all users. Start with $\beta=0$ (pure autoencoder) and slowly increase.

In [ ]:
def compute_metrics(model, train_data_tensor, test_data_np, k=20):
    """Compute Recall@K and NDCG@K."""
    model.eval()
    with torch.no_grad():
        logits, _, _ = model(train_data_tensor)
        # Mask out training items
        logits[train_data_tensor > 0] = -float('inf')
        _, top_k = torch.topk(logits, k, dim=1)
        top_k = top_k.cpu().numpy()
    
    recalls = []
    ndcgs = []
    for u in range(test_data_np.shape[0]):
        true_items = set(np.where(test_data_np[u] > 0)[0])
        if len(true_items) == 0:
            continue
        pred_items = list(top_k[u])
        hits = [1 if item in true_items else 0 for item in pred_items]
        recalls.append(sum(hits) / min(len(true_items), k))
        # NDCG
        dcg = sum(h / np.log2(i + 2) for i, h in enumerate(hits))
        idcg = sum(1.0 / np.log2(i + 2) for i in range(min(len(true_items), k)))
        ndcgs.append(dcg / idcg if idcg > 0 else 0)
    
    return np.mean(recalls), np.mean(ndcgs)


def train_mult_vae(model, train_data, test_data, n_epochs=50, batch_size=256,
                   lr=1e-3, beta_max=0.3, anneal_steps=2000):
    """Train Mult-VAE with KL annealing."""
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    
    train_tensor = torch.FloatTensor(train_data).to(device)
    dataset = TensorDataset(train_tensor)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    
    history = {'loss': [], 'recon': [], 'kl': [], 'recall': [], 'ndcg': [], 'beta': []}
    global_step = 0
    
    for epoch in range(n_epochs):
        model.train()
        epoch_loss = 0
        epoch_recon = 0
        epoch_kl = 0
        n_batches = 0
        
        for (batch_x,) in loader:
            # KL annealing schedule
            beta = min(beta_max, beta_max * global_step / anneal_steps)
            
            logits, mu, logvar = model(batch_x)
            loss, recon, kl = mult_vae_loss(logits, batch_x, mu, logvar, beta=beta)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
            epoch_recon += recon.item()
            epoch_kl += kl.item()
            n_batches += 1
            global_step += 1
        
        # Evaluate every 5 epochs
        if (epoch + 1) % 5 == 0:
            recall, ndcg = compute_metrics(model, train_tensor, test_data, k=20)
            history['recall'].append(recall)
            history['ndcg'].append(ndcg)
            history['loss'].append(epoch_loss / n_batches)
            history['recon'].append(epoch_recon / n_batches)
            history['kl'].append(epoch_kl / n_batches)
            history['beta'].append(beta)
            print(f'Epoch {epoch+1:3d} | Loss: {epoch_loss/n_batches:.2f} | '
                  f'Recon: {epoch_recon/n_batches:.2f} | KL: {epoch_kl/n_batches:.2f} | '
                  f'beta: {beta:.3f} | Recall@20: {recall:.4f} | NDCG@20: {ndcg:.4f}')
    
    return history


# Train the model
model = MultVAE(n_items=n_items, hidden_dim=256, latent_dim=64, dropout=0.5).to(device)
history = train_mult_vae(model, train_data, test_data, n_epochs=50, beta_max=0.3)

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

epochs = list(range(5, 51, 5))

axes[0].plot(epochs, history['recon'], 'b-o', label='Reconstruction', markersize=4)
axes[0].plot(epochs, history['kl'], 'r-s', label='KL Divergence', markersize=4)
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Losses')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs, history['recall'], 'g-o', label='Recall@20', markersize=4)
axes[1].plot(epochs, history['ndcg'], 'm-s', label='NDCG@20', markersize=4)
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Metric')
axes[1].set_title('Recommendation Quality')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].plot(epochs, history['beta'], 'k-o', markersize=4)
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Beta')
axes[2].set_title('KL Annealing Schedule')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. RecVAE: Composite Prior and Alternating Training

**RecVAE** (Shenbin et al., 2020) improves upon Mult-VAE with two key innovations:

1. **Composite prior**: Instead of the standard $\mathcal{N}(0, \mathbf{I})$ prior, RecVAE uses a mixture of the standard prior and the previous epoch's approximate posterior:

$$p^*(\mathbf{z}) = \alpha \cdot \mathcal{N}(0, \mathbf{I}) + (1 - \alpha) \cdot q_{\phi_{\text{old}}}(\mathbf{z}|\mathbf{x})$$

2. **Alternating training**: The encoder and decoder are trained in alternating steps, similar to EM.

> **💡 Concept:** The composite prior acts as a "soft anchor" — it prevents the latent space from drifting too far from the previous solution while still allowing exploration.

In [ ]:
class RecVAE(nn.Module):
    """RecVAE with composite prior and alternating training.
    
    Reference: Shenbin et al., "RecVAE: A New Variational Autoencoder for
    Top-N Recommendations with Implicit Feedback", WSDM 2020.
    """
    
    def __init__(self, n_items, hidden_dim=256, latent_dim=64, dropout=0.5):
        super().__init__()
        self.latent_dim = latent_dim
        
        # Encoder
        self.encoder = nn.Sequential(
            nn.Linear(n_items, hidden_dim),
            nn.Tanh()
        )
        self.fc_mu = nn.Linear(hidden_dim, latent_dim)
        self.fc_logvar = nn.Linear(hidden_dim, latent_dim)
        self.dropout = nn.Dropout(dropout)
        
        # Decoder
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, n_items)
        )
        
        self._init_weights()
    
    def _init_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_normal_(m.weight)
                nn.init.zeros_(m.bias)
    
    def encode(self, x):
        x_norm = F.normalize(x, p=2, dim=1)
        h = self.encoder(self.dropout(x_norm))
        return self.fc_mu(h), self.fc_logvar(h)
    
    def reparameterize(self, mu, logvar):
        if self.training:
            std = torch.exp(0.5 * logvar)
            return mu + std * torch.randn_like(std)
        return mu
    
    def decode(self, z):
        return self.decoder(z)
    
    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar


def composite_kl_loss(mu, logvar, prior_mu=None, prior_logvar=None, alpha=0.5):
    """KL divergence with composite prior.
    
    Approximation: KL to mixture ~ alpha * KL(q||N(0,I)) + (1-alpha) * KL(q||q_old)
    """
    # Standard KL to N(0, I)
    kl_standard = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1)
    
    if prior_mu is not None and prior_logvar is not None:
        # KL to previous posterior
        kl_prior = -0.5 * torch.sum(
            1 + logvar - prior_logvar - (mu - prior_mu).pow(2) / prior_logvar.exp()
            - logvar.exp() / prior_logvar.exp(),
            dim=1
        )
        return (alpha * kl_standard + (1 - alpha) * kl_prior).mean()
    
    return kl_standard.mean()


def train_recvae(model, train_data, test_data, n_epochs=50, batch_size=256,
                 lr=1e-3, beta=0.3, alpha=0.5):
    """Train RecVAE with alternating encoder/decoder updates."""
    enc_params = list(model.encoder.parameters()) + list(model.fc_mu.parameters()) + list(model.fc_logvar.parameters())
    dec_params = list(model.decoder.parameters())
    
    enc_optimizer = torch.optim.Adam(enc_params, lr=lr)
    dec_optimizer = torch.optim.Adam(dec_params, lr=lr)
    
    train_tensor = torch.FloatTensor(train_data).to(device)
    dataset = TensorDataset(train_tensor)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)
    
    history = {'recall': [], 'ndcg': []}
    prior_mu_cache = None
    prior_logvar_cache = None
    
    for epoch in range(n_epochs):
        model.train()
        
        # Cache current encoder outputs as prior for next epoch
        if epoch > 0:
            model.eval()
            with torch.no_grad():
                prior_mu_cache, prior_logvar_cache = model.encode(train_tensor)
            model.train()
        
        for step, (batch_x,) in enumerate(loader):
            batch_idx = torch.arange(step * batch_size,
                                     min((step + 1) * batch_size, len(train_tensor)))
            
            # Get prior for this batch
            p_mu = prior_mu_cache[batch_idx] if prior_mu_cache is not None else None
            p_lv = prior_logvar_cache[batch_idx] if prior_logvar_cache is not None else None
            
            # Phase 1: Update encoder
            logits, mu, logvar = model(batch_x)
            log_softmax = F.log_softmax(logits, dim=1)
            recon = -torch.sum(batch_x * log_softmax, dim=1).mean()
            kl = composite_kl_loss(mu, logvar, p_mu, p_lv, alpha=alpha)
            loss = recon + beta * kl
            
            enc_optimizer.zero_grad()
            loss.backward()
            enc_optimizer.step()
            
            # Phase 2: Update decoder
            logits, mu, logvar = model(batch_x)
            log_softmax = F.log_softmax(logits, dim=1)
            recon = -torch.sum(batch_x * log_softmax, dim=1).mean()
            kl = composite_kl_loss(mu, logvar, p_mu, p_lv, alpha=alpha)
            loss = recon + beta * kl
            
            dec_optimizer.zero_grad()
            loss.backward()
            dec_optimizer.step()
        
        if (epoch + 1) % 10 == 0:
            recall, ndcg = compute_metrics(model, train_tensor, test_data, k=20)
            history['recall'].append(recall)
            history['ndcg'].append(ndcg)
            print(f'RecVAE Epoch {epoch+1:3d} | Recall@20: {recall:.4f} | NDCG@20: {ndcg:.4f}')
    
    return history


recvae_model = RecVAE(n_items=n_items, hidden_dim=256, latent_dim=64).to(device)
recvae_history = train_recvae(recvae_model, train_data, test_data, n_epochs=50)

## 6. MacridVAE: Disentangled User Intents

**MacridVAE** (Ma et al., 2019, Microsoft) introduces the idea that users have multiple **macro intents** (e.g., genres, categories) and the latent representation should disentangle them.

Key ideas:
- Learn $K$ prototype vectors representing macro intents
- Assign items to intents via attention
- Each intent has its own latent code: $\mathbf{z}_u = [\mathbf{z}_{u,1}, ..., \mathbf{z}_{u,K}]$
- Final prediction combines intent-specific predictions

$$p(x_i | \mathbf{z}_u) = \sum_{k=1}^{K} \alpha_{ik} \cdot p_k(x_i | \mathbf{z}_{u,k})$$

where $\alpha_{ik}$ is the attention weight of item $i$ to intent $k$.

In [ ]:
class MacridVAE(nn.Module):
    """MacridVAE: Disentangled Representation Learning for CF.
    
    Reference: Ma et al., "Learning Disentangled Representations for Recommendation", NeurIPS 2019.
    """
    
    def __init__(self, n_items, hidden_dim=256, latent_dim=32, n_intents=4, dropout=0.5):
        super().__init__()
        self.n_intents = n_intents
        self.latent_dim = latent_dim
        self.dropout = nn.Dropout(dropout)
        
        # Intent prototypes (learnable)
        self.intent_prototypes = nn.Parameter(torch.randn(n_intents, hidden_dim) * 0.01)
        
        # Item embedding for intent assignment
        self.item_embed = nn.Embedding(n_items, hidden_dim)
        
        # Per-intent encoder
        self.encoder = nn.Linear(n_items, hidden_dim)
        self.fc_mu = nn.ModuleList([nn.Linear(hidden_dim, latent_dim) for _ in range(n_intents)])
        self.fc_logvar = nn.ModuleList([nn.Linear(hidden_dim, latent_dim) for _ in range(n_intents)])
        
        # Per-intent decoder
        self.decoders = nn.ModuleList([
            nn.Sequential(
                nn.Linear(latent_dim, hidden_dim),
                nn.Tanh(),
                nn.Linear(hidden_dim, n_items)
            ) for _ in range(n_intents)
        ])
    
    def get_intent_assignments(self):
        """Compute item-to-intent soft assignments."""
        item_embeds = self.item_embed.weight  # (n_items, hidden_dim)
        # Cosine similarity between items and prototypes
        item_norm = F.normalize(item_embeds, dim=1)
        proto_norm = F.normalize(self.intent_prototypes, dim=1)
        sim = item_norm @ proto_norm.T  # (n_items, n_intents)
        return F.softmax(sim / 0.1, dim=1)  # Temperature-scaled softmax
    
    def forward(self, x):
        x_norm = F.normalize(x, p=2, dim=1)
        h = torch.tanh(self.encoder(self.dropout(x_norm)))
        
        # Encode per intent
        all_mu, all_logvar, all_logits = [], [], []
        for k in range(self.n_intents):
            mu_k = self.fc_mu[k](h)
            logvar_k = self.fc_logvar[k](h)
            if self.training:
                z_k = mu_k + torch.exp(0.5 * logvar_k) * torch.randn_like(mu_k)
            else:
                z_k = mu_k
            logits_k = self.decoders[k](z_k)
            all_mu.append(mu_k)
            all_logvar.append(logvar_k)
            all_logits.append(logits_k)
        
        # Combine via intent assignments
        assignments = self.get_intent_assignments()  # (n_items, n_intents)
        combined_logits = torch.zeros_like(all_logits[0])
        for k in range(self.n_intents):
            combined_logits += all_logits[k] * assignments[:, k].unsqueeze(0)
        
        return combined_logits, all_mu, all_logvar


def macrid_loss(logits, x, all_mu, all_logvar, beta=0.2):
    """Loss for MacridVAE."""
    log_softmax = F.log_softmax(logits, dim=1)
    recon = -torch.sum(x * log_softmax, dim=1).mean()
    
    kl = 0
    for mu, logvar in zip(all_mu, all_logvar):
        kl += -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp(), dim=1).mean()
    kl /= len(all_mu)
    
    return recon + beta * kl, recon, kl


# Quick training run
macrid_model = MacridVAE(n_items=n_items, hidden_dim=128, latent_dim=32, n_intents=4).to(device)
optimizer = torch.optim.Adam(macrid_model.parameters(), lr=1e-3)
train_tensor = torch.FloatTensor(train_data).to(device)

macrid_model.train()
for epoch in range(30):
    logits, all_mu, all_logvar = macrid_model(train_tensor)
    loss, recon, kl = macrid_loss(logits, train_tensor, all_mu, all_logvar, beta=0.2)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if (epoch + 1) % 10 == 0:
        recall, ndcg = compute_metrics(macrid_model, train_tensor, test_data)
        print(f'MacridVAE Epoch {epoch+1} | Loss: {loss.item():.2f} | Recall@20: {recall:.4f} | NDCG@20: {ndcg:.4f}')

# Visualize intent assignments
macrid_model.eval()
assignments = macrid_model.get_intent_assignments().detach().cpu().numpy()
print(f'\nIntent assignment shape: {assignments.shape}')
print(f'Intent distribution (mean per intent): {assignments.mean(axis=0)}')

## 7. Beta-VAE: Controlling Disentanglement

The **Beta-VAE** approach (Higgins et al., 2017) for recommendation controls the trade-off between reconstruction quality and latent space structure by adjusting $\beta$:

$$\mathcal{L} = \mathbb{E}_{q}[\log p(\mathbf{x}|\mathbf{z})] - \beta \cdot \text{KL}(q(\mathbf{z}|\mathbf{x}) \| p(\mathbf{z}))$$

- $\beta < 1$: Emphasizes reconstruction, less regularized latent space
- $\beta = 1$: Standard VAE (original ELBO)
- $\beta > 1$: Stronger disentanglement, potentially worse reconstruction

For recommendations, the optimal $\beta$ is often $< 1$ (e.g., 0.2-0.5), as shown by Liang et al.

In [ ]:
# Experiment: sweep beta values
beta_values = [0.0, 0.1, 0.2, 0.3, 0.5, 0.8, 1.0]
beta_results = {}

for beta_val in beta_values:
    torch.manual_seed(42)
    model_b = MultVAE(n_items=n_items, hidden_dim=256, latent_dim=64, dropout=0.5).to(device)
    optimizer = torch.optim.Adam(model_b.parameters(), lr=1e-3)
    train_tensor = torch.FloatTensor(train_data).to(device)
    dataset = TensorDataset(train_tensor)
    loader = DataLoader(dataset, batch_size=256, shuffle=True)
    
    for epoch in range(40):
        model_b.train()
        for (batch_x,) in loader:
            logits, mu, logvar = model_b(batch_x)
            loss, _, _ = mult_vae_loss(logits, batch_x, mu, logvar, beta=beta_val)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    
    recall, ndcg = compute_metrics(model_b, train_tensor, test_data, k=20)
    beta_results[beta_val] = {'recall': recall, 'ndcg': ndcg}
    print(f'Beta={beta_val:.1f} | Recall@20: {recall:.4f} | NDCG@20: {ndcg:.4f}')

# Plot results
fig, ax = plt.subplots(1, 1, figsize=(8, 5))
betas = list(beta_results.keys())
recalls = [beta_results[b]['recall'] for b in betas]
ndcgs = [beta_results[b]['ndcg'] for b in betas]

ax.plot(betas, recalls, 'b-o', label='Recall@20', markersize=8)
ax.plot(betas, ndcgs, 'r-s', label='NDCG@20', markersize=8)
ax.set_xlabel('Beta (KL weight)', fontsize=12)
ax.set_ylabel('Metric Value', fontsize=12)
ax.set_title('Effect of Beta on Recommendation Quality', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.axvline(x=0.3, color='gray', linestyle='--', alpha=0.5, label='Typical optimal')
plt.tight_layout()
plt.show()

> **🔑 Pro Tip:** In practice, the optimal $\beta$ for recommendation is dataset-dependent but usually in the range $[0.1, 0.5]$. Values that are too high lead to "posterior collapse" where all users get similar recommendations, while $\beta=0$ reduces to a standard autoencoder without the benefits of the probabilistic framework.

## 8. Latent Space Visualization

In [ ]:
# Visualize the latent space for different beta values
from sklearn.decomposition import PCA

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for idx, beta_val in enumerate([0.0, 0.3, 1.0]):
    torch.manual_seed(42)
    model_viz = MultVAE(n_items=n_items, hidden_dim=256, latent_dim=64, dropout=0.5).to(device)
    optimizer = torch.optim.Adam(model_viz.parameters(), lr=1e-3)
    train_tensor = torch.FloatTensor(train_data).to(device)
    
    model_viz.train()
    for epoch in range(40):
        for i in range(0, len(train_tensor), 256):
            batch = train_tensor[i:i+256]
            logits, mu, logvar = model_viz(batch)
            loss, _, _ = mult_vae_loss(logits, batch, mu, logvar, beta=beta_val)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    
    model_viz.eval()
    with torch.no_grad():
        mu_all, _ = model_viz.encode(train_tensor[:500])
        z = mu_all.cpu().numpy()
    
    pca = PCA(n_components=2)
    z_2d = pca.fit_transform(z)
    
    # Color by number of interactions
    n_interactions = train_data[:500].sum(axis=1)
    scatter = axes[idx].scatter(z_2d[:, 0], z_2d[:, 1], c=n_interactions,
                                 cmap='viridis', alpha=0.5, s=10)
    axes[idx].set_title(f'Beta = {beta_val}', fontsize=12)
    axes[idx].set_xlabel('PC1')
    axes[idx].set_ylabel('PC2')
    plt.colorbar(scatter, ax=axes[idx], label='# interactions')

plt.suptitle('Latent Space Structure vs Beta', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## Exercises

### 🏋️ Exercise 1: Implement Mult-VAE with a Deeper Encoder

Modify the Mult-VAE to use a 2-layer encoder (600 -> 200 -> latent) and compare performance.

In [ ]:
# TODO: Implement a deeper Mult-VAE variant
class DeepMultVAE(nn.Module):
    def __init__(self, n_items, hidden_dims=(600, 200), latent_dim=64, dropout=0.5):
        super().__init__()
        # TODO: Build a deeper encoder with hidden_dims layers
        # TODO: Build a symmetric decoder
        pass
    
    def encode(self, x):
        # TODO: Implement encoding
        pass
    
    def reparameterize(self, mu, logvar):
        # TODO: Implement reparameterization trick
        pass
    
    def decode(self, z):
        # TODO: Implement decoding
        pass
    
    def forward(self, x):
        # TODO: Full forward pass
        pass

# TODO: Train and compare with the single-layer Mult-VAE

### 🏋️ Exercise 2: Tune Beta with Validation

Implement a proper beta-tuning loop that uses a validation set (split from training) to select the optimal beta.

In [ ]:
# TODO: Implement beta tuning with proper validation
def tune_beta(train_data, test_data, beta_candidates, n_epochs=30):
    """
    TODO:
    1. Split train_data into train/validation (80/20)
    2. For each beta candidate, train Mult-VAE and evaluate on validation
    3. Select best beta based on validation NDCG@20
    4. Report final test performance with best beta
    """
    pass

# beta_candidates = [0.05, 0.1, 0.15, 0.2, 0.25, 0.3, 0.4, 0.5]
# best_beta = tune_beta(train_data, test_data, beta_candidates)

### 🏋️ Exercise 3: Implement KL Annealing Strategies

Compare linear annealing vs cyclical annealing schedules for Mult-VAE.

In [ ]:
# TODO: Implement different annealing schedules
def linear_anneal(step, total_steps, beta_max=1.0):
    """Linear annealing from 0 to beta_max."""
    # TODO: Implement
    pass

def cyclical_anneal(step, cycle_length=1000, beta_max=1.0, n_cycles=4, ratio=0.5):
    """Cyclical annealing (Fu et al., 2019).
    
    TODO:
    1. Divide training into n_cycles
    2. In each cycle, linearly anneal for 'ratio' fraction, then hold at beta_max
    """
    pass

# TODO: Train Mult-VAE with each schedule and compare results
# TODO: Plot the annealing schedules and corresponding training curves

### 🏋️ Exercise 4: RecVAE with Different Alpha Values

Experiment with different alpha values in RecVAE's composite prior and analyze the effect.

In [ ]:
# TODO: Sweep alpha values for RecVAE
# alpha_values = [0.0, 0.25, 0.5, 0.75, 1.0]
# For each alpha:
#   1. Train RecVAE
#   2. Record Recall@20 and NDCG@20
#   3. Plot results
#
# Note: alpha=1.0 means standard prior only, alpha=0.0 means old-posterior-only
# TODO: Implement and analyze

## Summary

In this notebook, we covered:

| Model | Key Innovation | Year | Reference |
|-------|---------------|------|-----------|
| **Mult-VAE** | Multinomial likelihood for implicit CF | 2018 | Liang et al. (WWW) |
| **RecVAE** | Composite prior + alternating training | 2020 | Shenbin et al. (WSDM) |
| **MacridVAE** | Disentangled macro/micro intents | 2019 | Ma et al. (NeurIPS) |
| **Beta-VAE** | Controllable disentanglement via beta | 2017+ | Higgins et al. (ICLR) |

**Key Takeaways:**
1. The multinomial likelihood is a natural choice for implicit feedback
2. KL annealing is critical for preventing posterior collapse
3. Beta < 1 often works best for recommendation (unlike image generation)
4. Disentangled representations can capture multi-faceted user interests